week6 lab

In [ ]:
from datetime import datetime, timedelta

import cmcrameri as cmc  # noqa: F401
import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import pystac_client
import rioxarray  # noqa: F401
import xarray as xr
from odc.geo.geobox import GeoBox
from shapely.geometry import Polygon
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from skimage.filters import threshold_otsu


Part 1: Data Aquisiton, Preprocessing, and Feature Engineering

In [ ]:
# Define spatial and temporal extent for Vienna
dx = 0.0006  # ~60m resolution
epsg = 4326

# Vienna area bounds
vienna_bounds = (16.32, 47.86, 16.9, 48.407)

# Temporal extent: March 2020
start_date = datetime(year=2020, month=3, day=1)
end_date = start_date + timedelta(days=30)

date_query = f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}"

# Search for Sentinel-2 data
stac_client = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")

vienna_items = stac_client.search(
    bbox=vienna_bounds,
    collections=["sentinel-2-l2a"],
    datetime=date_query,
    query={"eo:cloud_cover": {"lt": 50}},  # skip heavily cloudy scenes
    limit=100,
).item_collection()

print(f"Vienna: {len(vienna_items)} scenes found")